In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy rustworkx
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# การลดข้อผิดพลาดในระดับ Utility-scale ด้วย Probabilistic Error Amplification

import TutorialFeedback from '@site/src/components/TutorialFeedback';





*ประมาณการใช้งาน: 14 นาทีบนโปรเซสเซอร์ Heron r3 (หมายเหตุ: นี่เป็นการประมาณเท่านั้น ระยะเวลาจริงอาจแตกต่างกัน)*
## ผลการเรียนรู้
หลังจากผ่านบทแนะนำนี้ ผู้ใช้ควรเข้าใจ:
- ทฤษฎีเบื้องหลัง *zero-noise extrapolation* (ZNE) วิธีต่างๆ ในการขยาย noise และเหตุใด *probabilistic error amplification* (PEA) จึงเป็นที่นิยมสำหรับการทดลองระดับ utility-scale
- วิธีใช้งาน ZNE ร่วมกับ PEA ในทางปฏิบัติโดยใช้ Qiskit

## ข้อกำหนดเบื้องต้น
เราแนะนำให้ผู้ใช้คุ้นเคยกับหัวข้อต่อไปนี้ก่อนผ่านบทแนะนำนี้:
- [บทเรียน Error mitigation](/learning/courses/utility-scale-quantum-computing/error-mitigation) ของหลักสูตร *Utility-scale quantum computing* สำหรับความรู้พื้นฐานเกี่ยวกับการใช้งาน error mitigation ใน Qiskit
- [บทเรียน Utility-I](/learning/courses/utility-scale-quantum-computing/utility-i) ของหลักสูตร *Utility-scale quantum computing* สำหรับพื้นหลังเพิ่มเติมเกี่ยวกับการทดลองระดับ utility-scale ที่ใช้เป็นตัวอย่างในบทแนะนำนี้
## พื้นหลัง
บทแนะนำนี้สาธิตวิธีรันการทดลองลดข้อผิดพลาดในระดับ utility-scale ด้วย Qiskit Runtime โดยใช้เวอร์ชันทดลองของ *zero-noise extrapolation* (ZNE) ร่วมกับ *probabilistic error amplification* (PEA)

![kim\_nature\_fig.png](https://quantum.cloud.ibm.com/docs/images/tutorials/utility-scale-error-mitigation-with-probabilistic-error-amplification/e1e67c34-9d4d-4a88-9340-f0b2f3676770.avif)
**อ้างอิง**: Y. Kim et al. *Evidence for the utility of quantum computing before fault tolerance.* [Nature 618.7965 (2023)](https://www.nature.com/articles/s41586-023-06096-3)
### Zero-noise extrapolation (ZNE)
Zero-noise extrapolation (ZNE) เป็นเทคนิคการลดข้อผิดพลาดที่ขจัดผลกระทบของ *ค่า noise ที่ไม่รู้จัก* ในระหว่างการรัน Circuit โดยที่ค่านั้นสามารถถูกปรับขนาดได้ใน *วิธีที่รู้จัก*

เทคนิคนี้สมมติว่าค่าคาดหวังเปลี่ยนแปลงตาม noise ด้วยฟังก์ชันที่รู้จัก

$$
\langle A(\lambda) \rangle = \langle A(0) \rangle + \sum_{k=0}^{m} a_k \lambda^k + R
$$

โดยที่ $\lambda$ เป็นพารามิเตอร์ที่แทนความแรงของ noise ซึ่งสามารถถูกขยายได้

เราสามารถใช้งาน ZNE ตามขั้นตอนต่อไปนี้:

1. ขยาย noise ของ Circuit สำหรับค่าตัวประกอบ noise หลายค่า $\lambda_1, \lambda_2, ... $
2. รัน Circuit ที่ถูกขยาย noise ทุกตัวเพื่อวัด $\langle A(\lambda_1)\rangle, ...$
3. Extrapolate กลับไปยังขีดจำกัดที่ไม่มี noise $\langle A(0)\rangle$

![zne\_stages.png](https://quantum.cloud.ibm.com/docs/images/tutorials/utility-scale-error-mitigation-with-probabilistic-error-amplification/5e63d706-82d8-4212-b802-c9191ce53341.avif)
#### การขยาย noise สำหรับ ZNE
ความท้าทายหลักในการใช้งาน ZNE ให้ประสบความสำเร็จคือการมีโมเดล noise ที่แม่นยำสำหรับค่าคาดหวัง และการขยาย noise ในวิธีที่รู้จัก

มีสามวิธีทั่วไปในการขยาย noise สำหรับ ZNE

| **Pulse stretching**                                                                                                                                                                               | **Gate folding**                                                                                                                                                                               | **Probabilistic error amplification**                                                                                                                                                |
| -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| ปรับขนาดระยะเวลาของ pulse ผ่านการ calibration                                                                                                                                                      | ทำซ้ำ Gate ในวงรอบ identity $U\mapsto U(U^{-1}U)^{\lambda-1}/2$                                                                                                                               | เพิ่ม noise ผ่านการ sampling Pauli channels                                                                                                                                          |
| ![zne\_pulse\_stretching.png](../docs/images/tutorials/utility-scale-error-mitigation-with-probabilistic-error-amplification/83188b57-e88f-43a1-a7bd-29327f46ecf5.avif) | ![zne\_gate\_folding.png](../docs/images/tutorials/utility-scale-error-mitigation-with-probabilistic-error-amplification/e1358d08-2632-4fd2-bf0f-f9384a2d3340.avif) | ![zne\_pea.png](../docs/images/tutorials/utility-scale-error-mitigation-with-probabilistic-error-amplification/3d69d5bd-70e5-4eeb-aa02-fc0a62043010.avif) |
| Kandala et al. Nature (2019)                                                                                                                                                                       | Shultz et al. PRA (2022)                                                                                                                                                                       | Li & Benjamin PRX (2017)                                                                                                                                                             |
สำหรับการทดลองระดับ utility-scale นั้น *probabilistic error amplification* (PEA) น่าสนใจที่สุด

* Pulse stretching สมมติว่า noise ของ Gate เป็นสัดส่วนกับระยะเวลา ซึ่งโดยทั่วไปไม่เป็นความจริง และการ calibration ก็มีต้นทุนสูง
* Gate folding ต้องการค่าตัวประกอบการยืดขนาดที่มาก ซึ่งจำกัด depth ของ Circuit ที่สามารถรันได้อย่างมาก
* PEA สามารถใช้กับ Circuit ใดก็ได้ที่สามารถรันด้วยค่า noise ดั้งเดิม ($\lambda=1$) แต่ต้องการการเรียนรู้โมเดล noise

### เรียนรู้โมเดล noise สำหรับ PEA
PEA สมมติโมเดล noise แบบ layer-based เดียวกับ *probabilistic error cancellation* (PEC) แต่หลีกเลี่ยงค่าใช้จ่ายจากการ sampling ที่เพิ่มขึ้นแบบ exponential ตาม noise ของ Circuit

| **ขั้นตอนที่ 1**                                                                                                                                                                                  | **ขั้นตอนที่ 2**                                                                                                                                                                               | **ขั้นตอนที่ 3**                                                                                                                                                                                |
| ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| Pauli twirl layers ของ two-qubit Gates                                                                                                                                                            | ทำซ้ำ identity pairs ของ layers และเรียนรู้ noise                                                                                                                                             | คำนวณค่า fidelity (ข้อผิดพลาดสำหรับแต่ละ noise channel)                                                                                                                                        |
| ![pec\_pauli\_twirling.png](../docs/images/tutorials/utility-scale-error-mitigation-with-probabilistic-error-amplification/2eab5ff4-40fa-4a41-9f2c-74f5e22c4643.avif) | ![pec\_learn\_layer.png](../docs/images/tutorials/utility-scale-error-mitigation-with-probabilistic-error-amplification/8d0d64c3-65ad-4419-8ac9-4ec9633d39a0.avif) | ![pec\_curve\_fitting.png](../docs/images/tutorials/utility-scale-error-mitigation-with-probabilistic-error-amplification/c51bd42d-2463-4c78-807b-d284ca79296f.avif) |

**อ้างอิง**: E. van den Berg, Z. Minev, A. Kandala, and K. Temme, *Probabilistic error cancellation with sparse Pauli-Lindblad models on noisy quantum processors* [arXiv:2201.09866](https://arxiv.org/abs/2201.09866)
## ข้อกำหนด
ก่อนเริ่มบทแนะนำนี้ ตรวจสอบให้แน่ใจว่าได้ติดตั้งสิ่งต่อไปนี้แล้ว:

- Qiskit SDK v2.0 หรือใหม่กว่า พร้อมรองรับ [visualization](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.22 หรือใหม่กว่า (`pip install qiskit-ibm-runtime`)
## การตั้งค่า
ใน cell ด้านล่าง เรานำเข้าแพ็กเกจที่จำเป็นและสร้างฟังก์ชันช่วยเพื่อสร้าง Circuit สำหรับ Trotterized time evolution ของโมเดล Ising สนามขวาง 2 มิติที่สอดคล้องกับ topology ของ Backend

In [1]:
from __future__ import annotations
from collections.abc import Sequence
from collections import defaultdict
import numpy as np
import rustworkx
import matplotlib.pyplot as plt

from qiskit.circuit import QuantumCircuit, Parameter
from qiskit.circuit.library import CXGate, CZGate, ECRGate
from qiskit.providers import Backend
from qiskit.visualization import plot_error_map
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import PubResult

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator


"""Trotter circuit generation"""


def remove_qubit_couplings(
    couplings: Sequence[tuple[int, int]], qubits: Sequence[int] | None = None
) -> list[tuple[int, int]]:
    """Remove qubits from a coupling list.

    Args:
        couplings: A sequence of qubit couplings.
        qubits: Optional, the qubits to remove.

    Returns:
        The input couplings with the specified qubits removed.
    """
    if qubits is None:
        return couplings
    qubits = set(qubits)
    return [edge for edge in couplings if not qubits.intersection(edge)]


def coupling_qubits(
    *couplings: Sequence[tuple[int, int]],
    allowed_qubits: Sequence[int] | None = None,
) -> list[int]:
    """Return a sorted list of all qubits involved in one or more couplings lists.

    Args:
        couplings: one or more coupling lists.
        allowed_qubits: Optional, the allowed qubits to include. If None all
            qubits are allowed.

    Returns:
        The intersection of all qubits in the couplings and the allowed qubits.
    """
    qubits = set()
    for edges in couplings:
        for edge in edges:
            qubits.update(edge)
    if allowed_qubits is not None:
        qubits = qubits.intersection(allowed_qubits)
    return list(qubits)


def construct_layer_couplings(
    backend: Backend,
) -> list[list[tuple[int, int]]]:
    """Separate a coupling map into disjoint 2-qubit gate layers.

    Args:
        backend: A backend to construct layer couplings for.

    Returns:
        A list of disjoint layers of directed couplings for the input coupling map.
    """
    coupling_graph = backend.coupling_map.graph.to_undirected(
        multigraph=False
    )
    edge_coloring = rustworkx.graph_bipartite_edge_color(coupling_graph)

    layers = defaultdict(list)
    for edge_idx, color in edge_coloring.items():
        layers[color].append(
            coupling_graph.get_edge_endpoints_by_index(edge_idx)
        )
    layers = [sorted(layers[i]) for i in sorted(layers.keys())]

    return layers


def entangling_layer(
    gate_2q: str,
    couplings: Sequence[tuple[int, int]],
    qubits: Sequence[int] | None = None,
) -> QuantumCircuit:
    """Generating a entangling layer for the specified couplings.

    This corresponds to a Trotter layer for a ZZ Ising term with angle Pi/2.

    Args:
        gate_2q: The 2-qubit basis gate for the layer, should be "cx", "cz", or "ecr".
        couplings: A sequence of qubit couplings to add CX gates to.
        qubits: Optional, the physical qubits for the layer. Any couplings involving
            qubits not in this list will be removed. If None the range up to the largest
            qubit in the couplings will be used.

    Returns:
        The QuantumCircuit for the entangling layer.
    """
    # Get qubits and convert to set to order
    if qubits is None:
        qubits = range(1 + max(coupling_qubits(couplings)))
    qubits = set(qubits)

    # Mapping of physical qubit to virtual qubit
    qubit_mapping = {q: i for i, q in enumerate(qubits)}

    # Convert couplings to indices for virtual qubits
    indices = [
        [qubit_mapping[i] for i in edge]
        for edge in couplings
        if qubits.issuperset(edge)
    ]

    # Layer circuit on virtual qubits
    circuit = QuantumCircuit(len(qubits))

    # Get 2-qubit basis gate and pre and post rotation circuits
    gate2q = None
    pre = QuantumCircuit(2)
    post = QuantumCircuit(2)

    if gate_2q == "cx":
        gate2q = CXGate()
        # Pre-rotation
        pre.sdg(0)
        pre.z(1)
        pre.sx(1)
        pre.s(1)
        # Post-rotation
        post.sdg(1)
        post.sxdg(1)
        post.s(1)
    elif gate_2q == "ecr":
        gate2q = ECRGate()
        # Pre-rotation
        pre.z(0)
        pre.s(1)
        pre.sx(1)
        pre.s(1)
        # Post-rotation
        post.x(0)
        post.sdg(1)
        post.sxdg(1)
        post.s(1)
    elif gate_2q == "cz":
        gate2q = CZGate()
        # Identity pre-rotation
        # Post-rotation
        post.sdg([0, 1])
    else:
        raise ValueError(
            f"Invalid 2-qubit basis gate {gate_2q}, should be 'cx', 'cz', or 'ecr'"
        )

    # Add 1Q pre-rotations
    for inds in indices:
        circuit.compose(pre, qubits=inds, inplace=True)

    # Use barriers around 2-qubit basis gate to specify a layer for PEA noise learning
    circuit.barrier()
    for inds in indices:
        circuit.append(gate2q, (inds[0], inds[1]))
    circuit.barrier()

    # Add 1Q post-rotations after barrier
    for inds in indices:
        circuit.compose(post, qubits=inds, inplace=True)

    # Add physical qubits as metadata
    circuit.metadata["physical_qubits"] = tuple(qubits)

    return circuit


def trotter_circuit(
    theta: Parameter | float,
    layer_couplings: Sequence[Sequence[tuple[int, int]]],
    num_steps: int,
    gate_2q: str | None = "cx",
    backend: Backend | None = None,
    qubits: Sequence[int] | None = None,
) -> QuantumCircuit:
    """Generate a Trotter circuit for the 2D Ising

    Args:
        theta: The angle parameter for X.
        layer_couplings: A list of couplings for each entangling layer.
        num_steps: the number of Trotter steps.
        gate_2q: The 2-qubit basis gate to use in entangling layers.
            Can be "cx", "cz", "ecr", or None if a backend is provided.
        backend: A backend to get the 2-qubit basis gate from, if provided
            will override the basis_gate field.
        qubits: Optional, the allowed physical qubits to truncate the
            couplings to. If None the range up to the largest
            qubit in the couplings will be used.

    Returns:
        The Trotter circuit.
    """
    if backend is not None:
        try:
            basis_gates = backend.configuration().basis_gates
        except AttributeError:
            basis_gates = backend.basis_gates
        for gate in ["cx", "cz", "ecr"]:
            if gate in basis_gates:
                gate_2q = gate
                break

    # If no qubits, get the largest qubit from all layers and
    # specify the range so the same one is used for all layers.
    if qubits is None:
        qubits = range(1 + max(coupling_qubits(layer_couplings)))

    # Generate the entangling layers
    layers = [
        entangling_layer(gate_2q, couplings, qubits=qubits)
        for couplings in layer_couplings
    ]

    # Construct the circuit for a single Trotter step
    num_qubits = len(qubits)
    trotter_step = QuantumCircuit(num_qubits)
    trotter_step.rx(theta, range(num_qubits))
    for layer in layers:
        trotter_step.compose(layer, range(num_qubits), inplace=True)

    # Construct the circuit for the specified number of Trotter steps
    circuit = QuantumCircuit(num_qubits)
    for _ in range(num_steps):
        circuit.rx(theta, range(num_qubits))
        for layer in layers:
            circuit.compose(layer, range(num_qubits), inplace=True)

    circuit.metadata["physical_qubits"] = tuple(qubits)
    return circuit


"""Result visualization functions"""


def plot_trotter_results(
    pub_result: PubResult,
    angles: Sequence[float],
    plot_noise_factors: Sequence[float] | None = None,
    plot_extrapolator: Sequence[str] | None = None,
    exact: np.ndarray = None,
    close: bool = True,
):
    """Plot average magnetization from ZNE result data.
    Args:
        pub_result: The Estimator PubResult for the PEA experiment.
        angles: The Rx angle values for the experiment.
        plot_raw: If provided plot the unextrapolated data for the noise factors.
        plot_extrapolator: If provided plot all extrapolators, if False only plot
            the Automatic method.
        exact: Optional, the exact values to include in the plot. Should be a 1D
            array-like where the values represent exact magnetization.
        close: Close the Matplotlib figure before returning.
    Returns:
        The figure.
    """
    data = pub_result.data

    evs = data.evs
    num_qubits = evs.shape[0]
    num_params = evs.shape[1]
    angles = np.asarray(angles).ravel()
    if angles.shape != (num_params,):
        raise ValueError(
            f"Incorrect number of angles for input data {angles.size} != {num_params}"
        )

    # Take average magnetization of qubits and its standard error
    x_vals = angles / np.pi
    y_vals = np.mean(evs, axis=0)
    y_errs = np.std(evs, axis=0) / np.sqrt(num_qubits)

    fig, _ = plt.subplots(1, 1)

    # Plot auto method
    plt.errorbar(x_vals, y_vals, y_errs, fmt="o-", label="ZNE (automatic)")

    # Plot individual extrapolator results
    if plot_extrapolator:
        y_vals_extrap = np.mean(data.evs_extrapolated, axis=0)
        y_errs_extrap = np.std(data.evs_extrapolated, axis=0) / np.sqrt(
            num_qubits
        )
        for i, extrap in enumerate(plot_extrapolator):
            plt.errorbar(
                x_vals,
                y_vals_extrap[:, i, 0],
                y_errs_extrap[:, i, 0],
                fmt="s-.",
                alpha=0.5,
                label=f"ZNE ({extrap})",
            )

    # Plot raw results
    if plot_noise_factors:
        y_vals_raw = np.mean(data.evs_noise_factors, axis=0)
        y_errs_raw = np.std(data.evs_noise_factors, axis=0) / np.sqrt(
            num_qubits
        )
        for i, nf in enumerate(plot_noise_factors):
            plt.errorbar(
                x_vals,
                y_vals_raw[:, i],
                y_errs_raw[:, i],
                fmt="d:",
                alpha=0.5,
                label=f"Raw (nf={nf:.1f})",
            )

    # Plot exact data
    if exact is not None:
        plt.plot(x_vals, exact, "--", color="black", alpha=0.5, label="Exact")

    plt.ylim(-0.1, 1.2)
    plt.xlabel("θ/π")
    plt.ylabel(r"$\overline{\langle Z \rangle}$")
    plt.legend()
    plt.title(
        f"Error Mitigated Average Magnetization for Rx(θ) [{num_qubits}-qubit]"
    )
    if close:
        plt.close(fig)
    return fig


def plot_qubit_zne_data(
    pub_result: PubResult,
    angles: Sequence[float],
    qubit: int,
    noise_factors: Sequence[float],
    extrapolator: Sequence[str] | None = None,
    extrapolated_noise_factors: Sequence[float] | None = None,
    num_cols: int | None = None,
    close: bool = True,
):
    """Plot ZNE extrapolation data for specific virtual qubit
    Args:
        pub_result: The Estimator PubResult for the PEA experiment.
        angles: The Rx theta angles used for the experiment.
        qubit: The virtual qubit index to plot.
        noise_factors: the raw noise factors.
        extrapolator: The extrapolator metadata for multiple extrapolators.
        extrapolated_noise_factors: The noise factors used for extrapolation.
        num_cols: The number of columns for the generated subplots.
        close: Close the Matplotlib figure before returning.
    Returns:
        The Matplotlib figure.
    """
    data = pub_result.data

    evs_auto = data.evs[qubit]
    stds_auto = data.stds[qubit]
    evs_extrap = data.evs_extrapolated[qubit]
    stds_extrap = data.stds_extrapolated[qubit]
    evs_raw = data.evs_noise_factors[qubit]
    stds_raw = data.stds_noise_factors[qubit]

    num_params = evs_auto.shape[0]
    angles = np.asarray(angles).ravel()
    if angles.shape != (num_params,):
        raise ValueError(
            f"Incorrect number of angles for input data {angles.size} != {num_params}"
        )

    # Make a square subplot
    num_cols = num_cols or int(np.ceil(np.sqrt(num_params)))
    num_rows = int(np.ceil(num_params / num_cols))
    fig, axes = plt.subplots(
        num_rows, num_cols, sharex=True, sharey=True, figsize=(12, 5)
    )
    fig.suptitle(f"ZNE data for virtual qubit {qubit}")

    for pidx, ax in zip(range(num_params), axes.flat):
        # Plot auto extrapolated
        ax.errorbar(
            0,
            evs_auto[pidx],
            stds_auto[pidx],
            fmt="o",
            label="PEA (automatic)",
        )

        # Plot extrapolators
        if (
            extrapolator is not None
            and extrapolated_noise_factors is not None
        ):
            for i, method in enumerate(extrapolator):
                ax.errorbar(
                    extrapolated_noise_factors,
                    evs_extrap[pidx, i],
                    stds_extrap[pidx, i],
                    fmt="-",
                    alpha=0.5,
                    label=f"PEA ({method})",
                )

        # Plot raw
        ax.errorbar(
            noise_factors, evs_raw[pidx], stds_raw[pidx], fmt="d", label="Raw"
        )

        ax.set_yticks([0, 0.5, 1, 1.5, 2])
        ax.set_ylim(0, max(1, 1.1 * max(evs_auto)))

        ax.set_xticks([0, *noise_factors])
        ax.set_title(f"θ/π = {angles[pidx]/np.pi:.2f}")
        if pidx == 0:
            ax.set_ylabel(r"$\langle Z_{" + str(qubit) + r"} \rangle$")
        if pidx == num_params - 1:
            ax.set_xlabel("Noise Factor")
            ax.legend()
    plt.tight_layout()
    if close:
        plt.close(fig)
    return fig

## ตัวอย่างจำลองขนาดเล็ก
เราจะข้ามขั้นตอนนี้เนื่องจาก runtime error mitigation ไม่รองรับบน simulator

## ตัวอย่างฮาร์ดแวร์ขนาดใหญ่
### ขั้นตอนที่ 1: แมป input แบบ classical ไปยังปัญหาเชิงควอนตัม
#### สร้าง Circuit โมเดล Ising แบบมีพารามิเตอร์
##### กำหนด Backend
ขั้นแรก เลือก Backend ที่จะใช้รัน การสาธิตนี้รันบน Backend ขนาด 127 Qubit แต่สามารถปรับเปลี่ยนเป็น Backend อื่นที่ใช้งานได้

In [2]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
backend

<IBMBackend('ibm_fez')>

##### Define entangling layer couplings
To implement the Trotterized Ising simulation, define three layers of two-qubit gate couplings for the device, to be repeated at each of the Trotter steps. These define the three twirled layers you need to learn the noise for to implement mitigation.

In [3]:
layer_couplings = construct_layer_couplings(backend)
for i, layer in enumerate(layer_couplings):
    print(f"Layer {i}:\n{layer}\n")

Layer 0:
[(2, 3), (4, 5), (6, 7), (8, 9), (10, 11), (12, 13), (14, 15), (16, 23), (18, 31), (19, 35), (20, 21), (25, 37), (26, 27), (28, 29), (33, 39), (36, 41), (38, 49), (42, 43), (45, 46), (47, 57), (51, 52), (53, 54), (56, 63), (58, 71), (59, 75), (61, 62), (64, 65), (66, 67), (68, 69), (72, 73), (76, 81), (79, 93), (82, 83), (84, 85), (86, 87), (88, 89), (91, 98), (94, 95), (97, 107), (99, 115), (100, 101), (102, 103), (105, 117), (108, 109), (110, 111), (113, 114), (116, 121), (118, 129), (123, 136), (124, 125), (126, 127), (130, 131), (132, 133), (135, 139), (138, 151), (142, 143), (144, 145), (146, 147), (152, 153), (154, 155)]

Layer 1:
[(0, 1), (3, 16), (5, 6), (7, 8), (11, 18), (13, 14), (17, 27), (21, 22), (23, 24), (25, 26), (29, 38), (30, 31), (32, 33), (34, 35), (39, 53), (41, 42), (43, 56), (44, 45), (47, 48), (49, 50), (51, 58), (54, 55), (57, 67), (60, 61), (62, 63), (65, 66), (69, 78), (70, 71), (73, 79), (74, 75), (77, 85), (80, 81), (83, 84), (87, 97), (89, 90), (9

##### กำหนด couplings ของ entangling layer
เพื่อใช้งาน Trotterized Ising simulation กำหนดสาม layers ของ two-qubit Gate couplings สำหรับอุปกรณ์ ซึ่งจะถูกทำซ้ำในแต่ละ Trotter step สิ่งเหล่านี้กำหนดสาม twirled layers ที่ต้องเรียนรู้ noise เพื่อใช้งานการลดข้อผิดพลาด

In [4]:
# Plot gate error map
# NOTE: These can change over time, so your results may look different
plot_error_map(backend)

<Image src="../docs/images/tutorials/probabilistic-error-amplification/extracted-outputs/fccef708-0.avif" alt="Output of the previous code cell" />

In [5]:
bad_qubits = {
    32,
    33,
    71,
    72,
    73,
    102,
    103,
}  # qubits removed based on high coupling error (1.00)
good_qubits = list(set(range(backend.num_qubits)).difference(bad_qubits))
print("Physical qubits:\n", good_qubits)

Physical qubits:
 [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155]


##### ลบ Qubit ที่มีปัญหา
ดู coupling map ของ Backend และตรวจสอบว่า Qubit ใดเชื่อมต่อกับ couplings ที่มีข้อผิดพลาดสูง ลบ Qubit "ที่มีปัญหา" เหล่านี้ออกจากการทดลอง

In [6]:
num_steps = 6
theta = Parameter("theta")
circuit = trotter_circuit(
    theta, layer_couplings, num_steps, qubits=good_qubits, backend=backend
)

#### Create a list of parameter values to be assigned later

In [7]:
num_params = 12

# 12 parameter values for Rx between [0, pi/2].
# Reshape to outer product broadcast with observables
parameter_values = np.linspace(0, np.pi / 2, num_params).reshape(
    (num_params, 1)
)
num_params = parameter_values.size

### Step 2: Optimize problem for quantum hardware execution

#### ISA circuit
Before running the circuit on hardware, optimize it for hardware execution. This process involves a few steps:

* Pick a qubit layout that maps the virtual qubits of your circuit to physical qubits on the hardware.
* Insert swap gates as needed to route interactions between qubits that are not connected.
* Translate the gates in our circuit to [Instruction Set Architecture (ISA)](/docs/guides/transpile#instruction-set-architecture) instructions that can directly be executed on the hardware.
* Perform circuit optimizations to minimize the circuit depth and gate count.

Although the transpiler built into Qiskit can perform all of these steps, this tutorial demonstrates building the utility-scale Trotter circuit in a ground-up fashion. Select the good physical qubits and define entangling layers on connected qubit pairs from those selected qubits. Nonetheless, you still need to translate non-ISA gates in the circuit and avail any circuit optimization offered by the transpiler.

Transpile your circuit for the chosen backend by creating a pass manager and then running the pass manager on the circuit. Also, fix the initial layout of the circuit to the already selected `good_qubits`. An easy way to create a pass manager is to use the [`generate_preset_pass_manager`](/docs/api/qiskit/qiskit.transpiler.generate_preset_pass_manager) function. See [Transpile with pass managers](/docs/guides/transpile-with-pass-managers) for a more detailed explanation of transpiling with pass managers.

In [8]:
pm = generate_preset_pass_manager(
    backend=backend,
    initial_layout=good_qubits,
    layout_method="trivial",
    optimization_level=1,
)

isa_circuit = pm.run(circuit)

##### การสร้าง Trotter Circuit หลัก

In [9]:
observables = []
num_qubits = len(good_qubits)
for q in range(num_qubits):
    observables.append(
        SparsePauliOp("I" * (num_qubits - q - 1) + "Z" + "I" * q)
    )

#### สร้างรายการค่าพารามิเตอร์เพื่อกำหนดในภายหลัง

In [10]:
isa_observables = [
    [obs.apply_layout(layout=isa_circuit.layout)] for obs in observables
]

### ขั้นตอนที่ 2: ปรับปรุง problem สำหรับการรันบนฮาร์ดแวร์ควอนตัม
#### ISA Circuit
ก่อนรัน Circuit บนฮาร์ดแวร์ ให้ปรับให้เหมาะกับการรันบนฮาร์ดแวร์ก่อน กระบวนการนี้มีหลายขั้นตอน:

* เลือก qubit layout ที่แมป virtual qubit ของ Circuit ไปยัง physical qubit บนฮาร์ดแวร์
* แทรก swap gate ตามที่จำเป็นเพื่อ route การโต้ตอบระหว่าง qubit ที่ไม่ได้เชื่อมต่อกัน
* แปลง Gate ใน Circuit ของเราเป็นคำสั่ง [Instruction Set Architecture (ISA)](/guides/transpile#instruction-set-architecture) ที่รันได้โดยตรงบนฮาร์ดแวร์
* ทำการ optimize Circuit เพื่อลด circuit depth และจำนวน gate

แม้ว่า Transpiler ที่ built-in ใน Qiskit จะทำทุกขั้นตอนเหล่านี้ได้ แต่ tutorial นี้จะแสดงการสร้าง utility-scale Trotter circuit แบบ ground-up โดยเลือก physical qubit ที่ดีและกำหนด entangling layer บน qubit pair ที่เชื่อมต่อกันจาก qubit ที่เลือกไว้ อย่างไรก็ตาม ยังต้องแปลง gate ที่ไม่ใช่ ISA ใน Circuit และใช้ประโยชน์จาก circuit optimization ที่ Transpiler มีให้

Transpile Circuit สำหรับ Backend ที่เลือกโดยสร้าง pass manager แล้วรัน pass manager บน Circuit ด้วย รวมถึงกำหนด initial layout ของ Circuit ให้กับ `good_qubits` ที่เลือกไว้แล้ว วิธีง่ายๆ ในการสร้าง pass manager คือใช้ฟังก์ชัน [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager) ดู [Transpile with pass managers](/guides/transpile-with-pass-managers) สำหรับคำอธิบายละเอียดเพิ่มเติมเกี่ยวกับการ transpile ด้วย pass manager

In [11]:
pub = (isa_circuit, isa_observables, parameter_values)

#### ISA Observables
ต่อไป สร้าง observable $\langle Z \rangle$ แบบ weight-1 ทั้งหมดสำหรับแต่ละ virtual qubit โดย padding จำนวน $\langle I \rangle$ ที่จำเป็น

In [12]:
# Experiment options
num_randomizations = 700
num_randomizations_learning = 40
max_batch_circuits = 3 * num_params
shots_per_randomization = 64
learning_pair_depths = [0, 1, 2, 4, 6, 12, 24]
noise_factors = [1, 1.3, 1.6]
extrapolated_noise_factors = np.linspace(0, max(noise_factors), 20)

# Base option formatting
options = {
    # Builtin resilience settings for ZNE
    "resilience": {
        "measure_mitigation": True,
        "zne_mitigation": True,
        # TREX noise learning configuration
        "measure_noise_learning": {
            "num_randomizations": num_randomizations_learning,
            "shots_per_randomization": 1024,
        },
        # PEA noise model configuration
        "layer_noise_learning": {
            "max_layers_to_learn": 3,
            "layer_pair_depths": learning_pair_depths,
            "shots_per_randomization": shots_per_randomization,
            "num_randomizations": num_randomizations_learning,
        },
        "zne": {
            "amplifier": "pea",
            "noise_factors": noise_factors,
            "extrapolator": ("exponential", "linear"),
            "extrapolated_noise_factors": extrapolated_noise_factors.tolist(),
        },
    },
    # Randomization configuration
    "twirling": {
        "num_randomizations": num_randomizations,
        "shots_per_randomization": shots_per_randomization,
        "strategy": "active-circuit",
    },
    # Optional Dynamical Decoupling (DD)
    "dynamical_decoupling": {"enable": True, "sequence_type": "XY4"},
    # Job tag
    "environment": {"job_tags": ["TUT_PEA"]},
}

กระบวนการ transpilation ได้แมป virtual qubit ของ Circuit ไปยัง physical qubit บนฮาร์ดแวร์ ข้อมูลเกี่ยวกับ qubit layout ถูกเก็บไว้ใน attribute `layout` ของ Circuit ที่ผ่านการ transpile แล้ว observable ของเราก็ถูกนิยามในรูปของ virtual qubit เช่นกัน ดังนั้นต้องนำ layout นี้ไปใช้กับ observable ด้วย โดยทำผ่าน method `apply_layout` ของ `SparsePauliOp`

สังเกตว่าแต่ละ observable ถูกห่อด้วย list ใน code block ต่อไปนี้ ทำเพื่อ *broadcast* กับ parameter value เพื่อให้แต่ละ qubit observable ถูกวัดสำหรับค่า theta แต่ละค่า ดูกฎ broadcasting สำหรับ primitive ได้ใน[เอกสารประกอบ primitives](/guides/primitives)

In [13]:
estimator = Estimator(mode=backend, options=options)
job = estimator.run([pub])
print(f"Job ID {job.job_id()}")

Job ID d7fa8oe2cugc739qbb10


In [14]:
job.status()

'DONE'

#### ตั้งค่าตัวเลือกของ Estimator
ต่อไปให้ตั้งค่าตัวเลือกของ `Estimator` ที่จำเป็นสำหรับรันการทดลองลดความผิดพลาด ซึ่งรวมถึงตัวเลือกสำหรับการเรียนรู้สัญญาณรบกวนของ entangling layers และการ extrapolation แบบ ZNE

เราใช้การตั้งค่าดังนี้:

In [15]:
primitive_result = job.result()

##### อธิบายตัวเลือก ZNE
ต่อไปนี้คือรายละเอียดเพิ่มเติมสำหรับตัวเลือกในสาขาทดลอง โปรดทราบว่าตัวเลือกและชื่อเหล่านี้ยังไม่ได้รับการยืนยัน และทุกอย่างอาจมีการเปลี่ยนแปลงก่อนปล่อยเวอร์ชันทางการ

* **amplifier**: วิธีที่ใช้ในการขยายสัญญาณรบกวนไปยัง noise factors ที่กำหนด
  ค่าที่รองรับได้แก่ `"gate_folding"` ซึ่งขยายโดยการทำซ้ำ two-qubit basis gates
  และ `"pea"` ซึ่งขยายโดยการสุ่มตัวอย่างแบบ probabilistic หลังจากเรียนรู้โมเดลสัญญาณรบกวนแบบ Pauli-twirled
  สำหรับ layers ของ twirled two-qubit basis gates นอกจากนี้ยังมี `"gate_folding_front"` และ `"gate_folding_back"` ซึ่งอธิบายไว้ใน [API documentation](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options#amplifier)
* **extrapolated\_noise\_factors**: ระบุค่า noise factor หนึ่งค่าหรือมากกว่าที่ต้องการประเมินโมเดล extrapolated
  ถ้าเป็น sequence ของค่า ผลลัพธ์ที่ได้จะเป็นแบบ array-valued โดย noise factor ที่ระบุจะถูกประเมินสำหรับโมเดล extrapolation ค่า 0 หมายถึง zero-noise extrapolation

#### รันการทดลอง

In [16]:
pub_result = primitive_result[0]

print(
    f"{pub_result.data.evs.shape=}\n"
    f"{pub_result.data.evs_extrapolated.shape=}\n"
    f"{pub_result.data.evs_noise_factors.shape=}\n"
)

pub_result.data.evs.shape=(149, 12)
pub_result.data.evs_extrapolated.shape=(149, 12, 2, 20)
pub_result.data.evs_noise_factors.shape=(149, 12, 3)



Several metadata fields are also available in the `PrimitiveResult`. The metadata includes

* `resilience/zne/noise_factors`: The raw noise factors
* `resilience/zne/extrapolator`: The extrapolators used for each result

In [17]:
primitive_result.metadata

{'dynamical_decoupling': {'enable': True,
  'sequence_type': 'XY4',
  'extra_slack_distribution': 'middle',
  'scheduling_method': 'alap'},
 'twirling': {'enable_gates': True,
  'enable_measure': True,
  'num_randomizations': 700,
  'shots_per_randomization': 64,
  'interleave_randomizations': True,
  'strategy': 'active-circuit'},
 'resilience': {'measure_mitigation': True,
  'zne_mitigation': True,
  'pec_mitigation': False,
  'zne': {'noise_factors': [1.0, 1.3, 1.6],
   'extrapolator': ['exponential', 'linear'],
   'extrapolated_noise_factors': [0.0,
    0.08421052631578947,
    0.16842105263157894,
    0.25263157894736843,
    0.3368421052631579,
    0.42105263157894735,
    0.5052631578947369,
    0.5894736842105263,
    0.6736842105263158,
    0.7578947368421053,
    0.8421052631578947,
    0.9263157894736842,
    1.0105263157894737,
    1.0947368421052632,
    1.1789473684210525,
    1.263157894736842,
    1.3473684210526315,
    1.431578947368421,
    1.5157894736842106,
    1.

The `PubResult` object has additional resilience metadata about the learned noise models used in mitigation.

In [18]:
# Print learned layer noise metadata
for field, value in pub_result.metadata["resilience"]["layer_noise"].items():
    print(f"{field}: {value}")

noise_overhead: 9.2584227461744e+229
total_mitigated_layers: 18
unique_mitigated_layers: 3
unique_mitigated_layers_noise_overhead: [2.0713004613510885e+36, 10.600275591731494, 9.687147432958504]


In [ ]:
# Exact data computed using the methods described in the original reference
# Y. Kim et al. "Evidence for the utility of quantum computing before fault tolerance" (Nature 618,
# 500–505 (2023)) Directly used here for brevity
exact_data = np.array(
    [
        1,
        0.9899,
        0.9531,
        0.8809,
        0.7536,
        0.5677,
        0.3545,
        0.1607,
        0.0539,
        0.0103,
        0.0012,
        0.0,
    ]
)

### ขั้นตอนที่ 4: ประมวลผลผลลัพธ์และแปลงเป็นรูปแบบ classical ที่ต้องการ
เมื่อการทดลองเสร็จสิ้น คุณสามารถดูผลลัพธ์ได้ โดยดึงค่า expectation values แบบ raw และแบบที่ลดความผิดพลาดแล้ว และนำมาเปรียบเทียบกับผลลัพธ์ที่แม่นยำ จากนั้นพล็อต expectation values ทั้งแบบที่ mitigated (extrapolated) และแบบ raw โดยเฉลี่ยทุก qubit สำหรับแต่ละพารามิเตอร์ สุดท้ายพล็อต expectation values ของ qubit แต่ละตัวตามที่เลือก

#### รูปร่างผลลัพธ์และ metadata ทั่วไป
อ็อบเจกต์ `PrimitiveResult` มีโครงสร้างคล้าย list ชื่อ `PubResult` เนื่องจากเราส่งเพียงหนึ่ง PUB ให้ estimator `PrimitiveResult` จึงมีอ็อบเจกต์ `PubResult` เพียงตัวเดียว

ค่า expectation values และ standard errors ของผลลัพธ์ PUB (primitive unified bloc) มีลักษณะเป็น array สำหรับ estimator jobs ที่ใช้ ZNE จะมี data fields หลายอย่างสำหรับค่า expectation values และ standard errors ใน `DataBin` container ของ `PubResult` เราจะกล่าวถึง data fields สำหรับค่า expectation values โดยสังเขปที่นี่ (data fields ที่คล้ายกันสำหรับ standard errors (`stds`) ก็มีให้ใช้เช่นกัน)

1. `pub_result.data.evs`: Expectation values ที่ตรงกับ zero noise (อิงจาก extrapolation ที่ดีที่สุดตาม heuristic)
    - แกนแรกคือ virtual qubit index สำหรับ observable $\langle Z_i\rangle$ ($124$ virtual-qubits/observables)
    - แกนที่สองคือ index ของค่าพารามิเตอร์สำหรับ $\theta$ ($12$ ค่าพารามิเตอร์)
2. `pub_result.data.evs_extrapolated`: Expectation values สำหรับ noise factors ที่ extrapolated สำหรับทุก extrapolator โดย array นี้มีสองแกนเพิ่มเติม
    - แกนที่สามคือ index ของวิธี extrapolation ($2$ extrapolators ได้แก่ `exponential` และ `linear`)
    - แกนสุดท้ายคือ index ของ `extrapolated_noise_factors` ($20$ จุด extrapolation ตามที่ระบุในตัวเลือก)
3. `pub_result.data.evs_noise_factors`: Expectation values แบบ raw สำหรับแต่ละ noise factor
   - แกนที่สามคือ index ของ `noise_factors` แบบ raw ($3$ factors)

In [20]:
zne_metadata = primitive_result.metadata["resilience"]["zne"]
# Plot Trotter simulation results
fig = plot_trotter_results(
    pub_result,
    parameter_values,
    plot_extrapolator=zne_metadata["extrapolator"],
    plot_noise_factors=zne_metadata["noise_factors"],
    exact=exact_data,
)
display(fig)

<Image src="../docs/images/tutorials/probabilistic-error-amplification/extracted-outputs/e466736a-0.avif" alt="Output of the previous code cell" />

While the noisy (noise factor `nf=1.0`) values show high deviation from exact values, the mitigated values are close to exact values, demonstrating the utility of the PEA-based mitigation technique.

### Plot extrapolation results for individual qubits

Finally, the following code creates a plot to show the extrapolation curves for different values of theta on a specific qubit.

In [21]:
virtual_qubit = 1
plot_qubit_zne_data(
    pub_result=pub_result,
    angles=parameter_values,
    qubit=virtual_qubit,
    noise_factors=zne_metadata["noise_factors"],
    extrapolator=zne_metadata["extrapolator"],
    extrapolated_noise_factors=zne_metadata["extrapolated_noise_factors"],
)

<Image src="../docs/images/tutorials/probabilistic-error-amplification/extracted-outputs/bea9695a-0.avif" alt="Output of the previous code cell" />

## Next steps
<Admonition type="tip" title="Recommendations">
If you found this work interesting, you might be interested in the following material:
- A [tutorial](/docs/tutorials/combine-error-mitigation-techniques) focused on combining error mitigation techniques.
- Detailed [documentation](/docs/guides/error-mitigation-and-suppression-techniques) on the error mitigation techniques available in Qiskit.
- Additional lessons covering utility-scale experiments: [Utility II](/learning/courses/utility-scale-quantum-computing/utility-ii) and [Utility III](/learning/courses/utility-scale-quantum-computing/utility-iii).
</Admonition>